# Installation

SysSimX supports Python 3.11, 3.12, and 3.13 on Windows, Linux, and macOS.

There are two ways to install SysSimX:

1. **Quick install** — for users who want to build and run co-simulations. One `pip install` command.
2. **Developer setup** — for contributors who work on the SysSimX source code itself.

If you are unsure, start with the quick install.

## Quick Install

Install the released package from PyPI with all supported adapters (FMI 2.0, OpenSim, and NGSolve structural dynamics):

```bash
pip install "syssimx[all]"
```

That is all you need to follow the tutorials in this documentation.

### Choosing what to install

If you do not need every backend, install only what you use:

| Command | What you get |
|---|---|
| `pip install syssimx` | Core framework only (no simulation backends) |
| `pip install "syssimx[fmu]"` | + FMI / Modelica support (FMPy, OMPython) |
| `pip install "syssimx[fem]"` | + transient structural-dynamics FEM support (NGSolve) |
| `pip install "syssimx[opensim]"` | + OpenSim 4.6+ Python API |
| `pip install "syssimx[all]"` | All of the above |

Extras can be combined, for example `pip install "syssimx[fmu,fem]"`.

```{tip}
Installing into a fresh virtual environment keeps your system Python clean:

    python -m venv .venv
    .venv\Scripts\Activate.ps1      # Windows PowerShell
    source .venv/bin/activate       # Linux / macOS
    pip install "syssimx[all]"
```

You do **not** need conda. OpenSim 4.6 ships PyPI wheels, so the plain `pip` install above covers all backends.

## Verify the Installation

Run the following cell in the environment you just installed into. `syssimx` and its core dependencies must be present; backend modules (`fmpy`, `ngsolve`, `opensim`) only appear if you installed the corresponding extra.

In [1]:
import importlib
import sys

print(sys.version)

for module_name in [
    "syssimx",
    "numpy",
    "scipy",
    "pandas",
    "matplotlib",
    "fmpy",
    "ngsolve",
    "opensim",
]:
    try:
        module = importlib.import_module(module_name)
    except ImportError as exc:
        print(f"{module_name}: not installed ({exc})")
        continue

    version = getattr(module, "__version__", "installed")
    print(f"{module_name}: {version}")

3.13.5 (main, Jun 12 2025, 12:42:35) [MSC v.1943 64 bit (AMD64)]
syssimx: 0.2.0
numpy: 2.5.1
scipy: 1.18.0
pandas: 3.0.3
matplotlib: 3.11.1
fmpy: 0.3.30
ngsolve: 6.2.2606
opensim: 4.6


## Developer Setup

The rest of this page is only relevant if you want to work on the SysSimX source code itself. It requires a clone of the [repository](https://github.com/FlorianFrech/SystemSimulation).

### Recommended: uv

For repository development, use [uv](https://docs.astral.sh/uv/) as the package manager. It reads `pyproject.toml`, creates the project environment, installs SysSimX editable, and locks the resolved development environment in `uv.lock`. The repository includes `.python-version`, so uv selects Python 3.13 by default.

Run the following commands in PowerShell from the repository root. The short environment path avoids Windows long-path failures that can occur while installing JupyterLab assets.

```powershell
uv --version
uv python install 3.13
$env:UV_PROJECT_ENVIRONMENT = "$env:USERPROFILE\.venvs\syssimx-313"
uv sync --python 3.13 --extra all
uv run python --version
```

For Linux or macOS:

```bash
uv python install 3.13
export UV_PROJECT_ENVIRONMENT="$HOME/.venvs/syssimx-313"
uv sync --python 3.13 --extra all
uv run python --version
```

`uv sync` installs SysSimX editable because the project has a build system. The `dev` dependency group is included by default, and `--extra all` adds the FMI, NGSolve structural-dynamics, and OpenSim optional adapters.

### Alternative: pip Editable Install

To develop without uv, create a virtual environment and install the repository in editable mode. Run this from the repository root:

```powershell
python -m venv "$env:USERPROFILE\.venvs\syssimx-313"
& "$env:USERPROFILE\.venvs\syssimx-313\Scripts\Activate.ps1"
python -m pip install --upgrade pip setuptools wheel
python -m pip install -e ".[full]"
```

The `full` extra combines all simulation backends (`all`) with the development tooling (`dev`: pytest, ruff, mypy, Jupyter, Sphinx).

### Register the Jupyter Kernel

To run the documentation notebooks against your development environment, register it as a notebook kernel. With uv:

```powershell
uv run python -m ipykernel install --user --name syssimx-313 --display-name "Python 3.13 (SysSimX)"
```

With an activated `venv`/`pip` environment, use the same command without `uv run`:

```powershell
python -m ipykernel install --user --name syssimx-313 --display-name "Python 3.13 (SysSimX)"
```

### Should I Still Use Conda?

You do not need conda for OpenSim 4.6. Use conda only if you want conda to manage the Python interpreter itself.

If you choose conda, keep the workflow simple: create a Python environment, activate it, and use `python -m pip install ...` for SysSimX and OpenSim. Avoid mixing old conda OpenSim builds with pip-installed NumPy/SciPy in the same environment.

### Package Management and Publishing (Maintainers)

Keep published runtime dependencies and optional extras in `pyproject.toml`. Use uv commands to update that metadata and refresh `uv.lock`:

```powershell
uv add "numpy"
uv add --optional opensim "opensim>=4.6"
uv add --dev pytest
uv lock
```

For releases, build artifacts with uv and publish them from `dist/`:

```powershell
uv sync --python 3.13 --extra all
uv run pytest tests
uv build --no-sources
uv publish --index testpypi
uv publish
```

`uv publish --index testpypi` uses the `testpypi` publishing index configured in `pyproject.toml`. Set `UV_PUBLISH_TOKEN` for token-based uploads, or use PyPI Trusted Publishing in CI.

## Troubleshooting

- If PowerShell blocks activation in a plain `venv` workflow, run `Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope CurrentUser` once and reopen the shell. The uv workflow does not require activation when commands are run through `uv run`.
- If a Windows install fails with a long-path error while installing JupyterLab, set `UV_PROJECT_ENVIRONMENT` to a short path such as `$env:USERPROFILE\.venvs\syssimx-313` or enable Windows Long Paths.
- If `uv sync` refuses to use an already activated environment, either deactivate it or pass `--active` intentionally.
- If `opensim` does not install, check that your Python version and platform have a matching OpenSim wheel on PyPI.